# Pilot calibration → freeze → auto / human decision (plan B)

**The one calibration notebook for every evaluator** — the former `analysis_direction.ipynb` and
`analysis_body_side.ipynb` are merged into it.

Run **once**, after every model's Pilot has been evaluated and labelled in `run_benchmark.ipynb`
(STEP 9 labels, STEP 11 saves `<Model>_pilot_results.json`).

| Step | What | Output |
|---|---|---|
| 1 | Load the Pilot results of all models | — |
| 2 | Agreement with Human Gold per requirement type (real labels) | table |
| 3 | Synthetic negatives: left/right swap, time reversal, frozen pose (needs the motion ZIPs) | table |
| 4 | Grid search of the thresholds of every evaluator on all models together | best thresholds |
| 4b | Inspect one requirement type: every disagreement with its evidence (for the evaluator's owner) | table |
| 5 | Leave-one-model-out check | table |
| 6 | Decide per type: **auto** (rules) or **human** (κ ≥ 0.6, n ≥ 10, ≥ 3 FAIL / 3 PASS, real agreement ≥ 0.8) | `evaluation_mode.json` |
| 7 | Freeze: code snippets for `CURRENT_THRESHOLDS`, download files | PR + tag `v1.0-pilot-frozen` |

Types that do not pass the gate are **not** improved — they are labelled by people in Main.
All evaluators used by `EVALUATION_CONFIG` are calibrated here, including Direction (body frame) and
Body Side. A's world-frame `TrajectoryEvaluator` is no longer the default and is not calibrated here.

## STEP 0 — Get the code from GitHub

Clones the repository (first run) or pulls the latest version, then imports `evaluation/common.py` and **every `evaluation/eval_*.py` automatically** — a new evaluator file is picked up without editing this notebook.

- `BRANCH = "main"` for normal use; set it to your branch name to test your work before it is merged.
- Private repository only: add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon, left sidebar).
- CPU runtime is enough for evaluation (no GPU needed).

In [ ]:
import importlib, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Soniaaaa-aa/t2m-capability-benchmark"
BRANCH = "main"                                   # or your feature branch
REPO_DIR = Path("/content/t2m-capability-benchmark")

def _git(*args, cwd=None):
    print("$ git", " ".join(args))
    subprocess.run(["git", *args], cwd=cwd, check=True)

url = REPO_URL
try:  # optional token for a private repository
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if not (REPO_DIR / ".git").exists():
    _git("clone", "--branch", BRANCH, url, str(REPO_DIR))
else:
    _git("fetch", "origin", cwd=REPO_DIR)
    _git("checkout", BRANCH, cwd=REPO_DIR)
    _git("pull", "origin", BRANCH, cwd=REPO_DIR)

EVAL_DIR = REPO_DIR / "evaluation"
if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))

import common
importlib.reload(common)                 # pick up changes after a pull
from common import *                     # settings, loaders, registry, gold helpers, runner
evaluator_modules = load_all_evaluators(EVAL_DIR)   # imports every eval_*.py

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
print("Framework commit     :", commit or "unknown")

import json
import validation as V
import finalize as F

def download(path):
    try:
        from google.colab import files
        files.download(str(path))
    except Exception as e:  # not in Colab
        print("(download skipped:", type(e).__name__, ")", path)

## STEP 1 — Load the Pilot results of all models

Put the `<Model>_pilot_results.json` files (run_benchmark STEP 11) into `RESULTS_DIR`, or upload them.

In [ ]:
RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
files_found = sorted(RESULTS_DIR.glob("*_pilot_results.json"))
if not files_found:
    from google.colab import files
    print("Upload the <Model>_pilot_results.json files of ALL models")
    for name, data in files.upload().items():
        (RESULTS_DIR / name).write_bytes(data)
    files_found = sorted(RESULTS_DIR.glob("*_pilot_results.json"))

rows = []
for f in files_found:
    payload = json.loads(f.read_text(encoding="utf-8"))
    rows += payload["results"]
    print(f"{f.name:45s} model={payload['metadata'].get('model')}  rows={len(payload['results'])}")
MODELS = sorted({r["model"] for r in rows})
print("Total rows:", len(rows), " models:", MODELS)
missing = sum(r.get("human_label") is None for r in rows)
if missing:
    print(f"⚠️ {missing} rows have no Human Gold label (labels/<Model>_pilot_human_gold_labels.json)")

## STEP 2 — Agreement with Human Gold (real labels only)

In [ ]:
real_report = V.agreement_report(rows)

## STEP 3 — Synthetic negatives

Real Pilot labels are mostly PASS, so κ cannot be measured without FAIL examples. Three transforms of
motions labelled PASS give requirements that **must** FAIL:

* left/right swap → body_side, turn_direction, left/right direction, left_/right_ target
* time reversal → 2-step order, forward/backward direction, turn_direction
* frozen first pose → every requirement (easy; report real-only agreement too)

Used only here, never in model scores. Needs each model's motion ZIP (asked for if missing).

In [ ]:
USE_SYNTHETIC_NEGATIVES = True
NEG_MODELS = MODELS            # or a subset, e.g. ["MoMADiff"]
neg_rows = []
if USE_SYNTHETIC_NEGATIVES:
    for model in NEG_MODELS:
        data = load_inputs(model, REPO_DIR, verbose=False)
        if data["human_gold_labels"] is None:
            print(model, ": no labels in labels/ — skipped")
            continue
        neg_cases, neg_labels = V.make_synthetic_negatives(
            data["evaluation_cases"], data["human_gold_labels"], Path("/content/synthetic_negatives") / model)
        neg_rows += run_all_evaluators(neg_cases, neg_labels)
        print(f"{model}: {len(neg_cases)} negative motions")
    print("\nAgreement on synthetic negatives (current thresholds):")
    V.agreement_report(neg_rows)

## STEP 4 — Grid search (real + synthetic, all models together)

Unlisted thresholds keep their provisional value. Ranking: κ, then agreement; ties → closest to the
plan's starting values.

In [ ]:
GRIDS = {
    "ActionEvaluator":          {"walk_min_duration_s": [0.5, 1.0, 1.5], "walk_min_distance_m": [0.3, 0.5, 0.8],
                                 "turn_min_deg": [30, 45, 60, 90]},
    "BodyFrameDirectionEvaluator": {"min_displacement": [0.25, 0.5, 0.75, 1.0]},
    "BodySideEvaluator":        {"min_activity": [0.3, 0.5, 0.7, 0.9, 1.1], "side_margin": [0.1, 0.2, 0.25, 0.3, 0.4, 0.5],
                                 "both_max_imbalance": [0.3, 0.5, 0.7]},
    "RotationEvaluator":        {"min_turn_deg": [20, 30, 45, 60, 75, 90]},
    "CountEvaluator":           {"tolerance": [0, 1]},
    "OrderEvaluator":           {"min_start_gap_frames": [1, 5, 10, 20]},
    "SimultaneousEvaluator":    {"min_overlap_ratio": [0.2, 0.3, 0.5, 0.7, 0.9]},
    "AttributeEvaluator":       {"speed_ratio": [1.1, 1.2, 1.3, 1.5], "min_walk_speed": [0.2, 0.3, 0.5]},
    "SpatialRelationEvaluator": {"target_max_dist_sw": [0.5, 0.75, 1.0, 1.25], "cross_min_sw": [0.0, 0.1, 0.25, 0.4]},
    "LimbGeometryEvaluator":    {"min_extent": [0.2, 0.35, 0.5, 0.7], "min_dominance": [1.0, 1.2, 1.5, 2.0]},
    "TorsoGeometryEvaluator":   {"min_tilt_deg": [10, 15, 20, 30], "min_dominance": [1.0, 1.5, 2.0]},
}
all_rows = rows + neg_rows
best = {}
for name, grid in GRIDS.items():
    cls = get_evaluator(name)
    b, table = V.grid_search(cls, all_rows, grid)
    top = table[0] if table else None
    if not top or top["n"] == 0:
        print(f"{name:26s} no labelled rows — keep provisional")
        continue
    best[name] = b
    changed = {k: v for k, v in b.items() if cls.CURRENT_THRESHOLDS.get(k) != v}
    ka = "-" if top["kappa"] is None else f"{top['kappa']:.2f}"
    print(f"{name:26s} n={top['n']:3d} kappa={ka} agree={top['agreement']:.2f}  change: {changed or 'none'}")

## STEP 4b — Inspect one requirement type (for its owner)

Set `TYPE` (e.g. `"body_side"`, `"direction"`, `"count"`). Lists every labelled requirement where the rule
(with the selected thresholds) and the human disagree, with the evidence numbers the rule used — the
replacement for the former per-evaluator analysis notebooks.

In [ ]:
TYPE = "body_side"                      # ← any requirement type
SHOW_ALL = False                        # True: also list agreements
cand = [r for r in F.redecide(all_rows, best) if r["requirement_type"] == TYPE
        and r.get("pass_fail") in ("PASS", "FAIL") and r.get("human_label") in ("PASS", "FAIL")]
shown = cand if SHOW_ALL else [r for r in cand if r["pass_fail"] != r["human_label"]]
print(f"{TYPE}: {len(cand)} labelled, {sum(r['pass_fail'] != r['human_label'] for r in cand)} disagreements")
SKIP = {"events", "turn_events", "groups", "best_combination", "left_detail", "right_detail", "all_limbs",
        "per_side", "steps", "dominant_turn", "components"}
for r in shown:
    ev = r.get("evidence") or {}
    brief = {k: (round(v, 3) if isinstance(v, float) else v) for k, v in ev.items()
             if k not in SKIP and not isinstance(v, (list, dict))}
    print(f"\n{r['model']:12s} {r['prompt_id']:8s} expected={r['expected_value']!s:14s} "
          f"rule={r['pass_fail']:4s} human={r['human_label']}")
    print("   reason  :", r.get("reason"))
    print("   evidence:", brief)

## STEP 5 — Leave-one-model-out

Fit on all models but one, test on the held-out one. A large drop means the threshold favours some models.

In [ ]:
if len(MODELS) < 2:
    print("Needs results of at least 2 models — skipped")
else:
    for name, grid in GRIDS.items():
        if name not in best:
            continue
        res = V.leave_one_model_out(get_evaluator(name), all_rows, grid)
        for held, r in res.items():
            ka = "-" if r["test_kappa"] is None else f"{r['test_kappa']:.2f}"
            print(f"{name:26s} held-out {held:14s} n={r['test_n']:3d} kappa={ka} agree={r['test_agreement']:.2f}")

## STEP 6 — auto / human per requirement type

The selected thresholds are applied to the saved evidence (no re-run), then each type is checked
against the gate (`finalize.DEFAULT_POLICY`). The decision is saved as `evaluation_mode.json`.

In [ ]:
POLICY = dict(F.DEFAULT_POLICY)          # change only with the whole team's agreement
real_final = F.redecide(rows, best)
neg_final = F.redecide(neg_rows, best)
mode = F.decide_modes(real_final, neg_final, POLICY)
F.print_modes(mode)

print("\nReal-label agreement with the selected thresholds (report this in the paper):")
V.agreement_report(real_final)

MODE_PATH = RESULTS_DIR / "evaluation_mode.json"
F.save_mode(mode, MODE_PATH, {"framework_commit": commit, "models": MODELS,
                              "synthetic_negatives": USE_SYNTHETIC_NEGATIVES}, frozen_thresholds=best)
print("\nSaved:", MODE_PATH)

## STEP 7 — Freeze

1. Copy each printed block into the evaluator's file (`CURRENT_THRESHOLDS`, `CURRENT_THRESHOLD_STATUS`).
2. Put `evaluation_mode.json` into `benchmark/` of the repository.
3. One Pull Request with both, reviewed by the team; after merging, tag the commit `v1.0-pilot-frozen`.
4. Main is run only from that tag (`run_main.ipynb`, `BRANCH = "v1.0-pilot-frozen"`). It warns if the
   thresholds in the code differ from `evaluation_mode.json`.

In [ ]:
FILES = {"ActionEvaluator": "eval_action.py", "RotationEvaluator": "eval_rotation.py",
         "BodyFrameDirectionEvaluator": "eval_trajectory_ext.py",
         "BodySideEvaluator": "eval_body_side.py",
         "CountEvaluator": "eval_temporal.py", "OrderEvaluator": "eval_temporal.py",
         "SimultaneousEvaluator": "eval_temporal.py", "AttributeEvaluator": "eval_trajectory_ext.py",
         "SpatialRelationEvaluator": "eval_spatial.py", "LimbGeometryEvaluator": "eval_limb.py",
         "TorsoGeometryEvaluator": "eval_limb.py"}
for name, th in best.items():
    print(f"# evaluation/{FILES[name]} — class {name}")
    print(f"CURRENT_THRESHOLDS = {th!r}")
    print('CURRENT_THRESHOLD_STATUS = "frozen_v1.0"\n')
download(MODE_PATH)